use python/neuralnetmodelling/savecompletemodel.ipynb to export the model to keras
then  

conda activate guitarmidi_nn311
and 

ext/frugally-deep/keras_export/convert_model.py python/neuralnetmodelling/model_dir/guitarmidi-model-and-weights.keras python/neuralnetmodelling/model_dir/guitarmidi-model-and-weights.json

To convert the frugally deep json

In [ ]:
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES,OUTPUT_DIM_ONSETS
import numpy as np
from model import build_cnn_model

print(INPUT_SHAPE)
cnn_model=build_cnn_model(INPUT_SHAPE,OUTPUT_DIM_NOTES,False)
cnn_model.load_weights('guitarmidi.weights.h5')
# cnn_model.save('guitarmidi-model-and-weights.weights.h5')
cnn_model.save('model_dir/guitarmidi-model-and-weights.keras')
# cnn_model.export('model_dir/')

(256, 312, 1)


In [13]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import Conv2D, Dense, BatchNormalization

def fuse_bn_to_dense_or_conv(layer, bn_layer):
    # Get Conv/Dense weights and bias
    W, b = layer.get_weights()
    if b is None:
        b = np.zeros(W.shape[-1] if isinstance(layer, Dense) else layer.filters)
    
    # Get BN params
    gamma, beta, mean, var = bn_layer.get_weights()
    epsilon = bn_layer.epsilon
    
    # Fuse parameters
    std = np.sqrt(var + epsilon)
    W_fused = W * (gamma / std).reshape((-1,) if isinstance(layer, Dense) else (1, 1, 1, -1))
    b_fused = beta + (b - mean) * gamma / std
    
    return [W_fused, b_fused]

def fuse_model(model):
    # Create new model layers list
    new_layers = []
    skip_next = False
    
    for i in range(len(model.layers)):
        if skip_next:
            skip_next = False
            continue
        
        layer = model.layers[i]
        
        # Check if next layer is BatchNormalization
        if i + 1 < len(model.layers) and isinstance(model.layers[i + 1], BatchNormalization):
            bn_layer = model.layers[i + 1]
            if isinstance(layer, (Conv2D, Dense)):
                # Fuse
                fused_weights = fuse_bn_to_dense_or_conv(layer, bn_layer)
                layer.set_weights(fused_weights)
                new_layers.append(layer)
                skip_next = True  # skip BN layer
            else:
                new_layers.append(layer)
        else:
            new_layers.append(layer)

    # Build a new model without BN layers (only fused layers)
    inputs = model.inputs
    x = inputs[0]
    for layer in new_layers:
        x = layer(x)
    fused_model = tf.keras.Model(inputs=inputs, outputs=x)
    return fused_model


# Fuse BN layers into Conv2D/Dense weights
fused_model = fuse_model(cnn_model)

# Save the fused model for inference
fused_model.save('model_dir/guitarmidi-model-and-weights.keras')
